In [ ]:
# BAGIAN A: (Apa itu Feature-Engine?)

# feature-engine adalah library Python. Isinya adalah transformer untuk membersihkan data dan membuat fitur baru, yang bisa digabung dengan pipeline Scikit-learn.

In [1]:
import pandas as pd
import numpy as np

# Impor 3 transformer dari library feature-engine
from feature_engine.imputation import CategoricalImputer
from feature_engine.encoding import RareLabelEncoder, OneHotEncoder

# Muat data untuk demo
data = pd.read_csv('data/train.csv')

In [ ]:
### Demo 1: `CategoricalImputer` (Mengisi Data Hilang)

# Problem: Model error kalau ketemu `NaN` (data kosong).
# Fungsi: Mengisi data `NaN` di kolom kategorial.

In [3]:
# BEFORE
print("--- BEFORE ---")
# Kita lihat kolom 'Alley' (gang), isinya banyak NaN
print(data['Alley'].value_counts(dropna=False)) 

# Gunakan transformer
# Kita bilang: isi semua NaN dengan string 'None'
imputer = CategoricalImputer(imputation_method='missing', fill_value='None', variables=['Alley'])
data_transformed = imputer.fit_transform(data)

# AFTER
print("\n--- AFTER ---")
# Sekarang NaN sudah hilang, diganti jadi 'None'
print(data_transformed['Alley'].value_counts(dropna=False))

--- BEFORE ---
Alley
NaN     1369
Grvl      50
Pave      41
Name: count, dtype: int64

--- AFTER ---
Alley
None    1369
Grvl      50
Pave      41
Name: count, dtype: int64


In [ ]:
# Demo 2: `RareLabelEncoder` (Mengelompokkan Kategori Langka)

# Problem: Kolom Neighborhood (perumahan) punya 25 nama. Ini terlalu banyak dan bisa bikin pusing model.
# Fungsi: Mengelompokkan kategori yang jarang muncul (misal, < 5% data) jadi satu grup bernama 'Rare'.

In [4]:
# BEFORE
print("--- BEFORE ---")
print(f"Jumlah kategori unik: {data['Neighborhood'].nunique()}")
print(data['Neighborhood'].value_counts().head(10)) # Tampilkan 10 terbanyak

# Gunakan transformer
# Kita bilang: gabung kategori yg munculnya < 5% (tol=0.05) jadi 'Rare'
rare_encoder = RareLabelEncoder(tol=0.05, n_categories=1, variables=['Neighborhood'])
data_transformed = rare_encoder.fit_transform(data)

# AFTER
print("\n--- AFTER ---")
print(f"Jumlah kategori unik: {data_transformed['Neighborhood'].nunique()}")
print(data_transformed['Neighborhood'].value_counts()) # Kategori 'langka' sudah digabung

--- BEFORE ---
Jumlah kategori unik: 25
Neighborhood
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
SawyerW     59
Name: count, dtype: int64

--- AFTER ---
Jumlah kategori unik: 10
Neighborhood
Rare       483
NAmes      225
CollgCr    150
OldTown    113
Edwards    100
Somerst     86
Gilbert     79
NridgHt     77
Sawyer      74
NWAmes      73
Name: count, dtype: int64


In [ ]:
### Demo 3: `OneHotEncoder` (Mengubah Teks jadi Angka)

# Problem: Model tidak bisa menghitung "Pave" atau "Grvl". Model butuh angka.
# Fungsi: Mengubah 1 kolom kategori jadi beberapa kolom 0/1 (biner).
# Dampak: Ini adalah langkah wajib sebelum data masuk ke model. Tanpa ini, model **tidak akan bisa berjalan** dan akurasi = 0. Dengan One-Hot, model bisa *mengevaluasi* dampak dari tiap kategori (misal: 'jalan Pave lebih mahal dari Grvl') dan meningkatkan akurasi.

In [5]:
from feature_engine.encoding import OneHotEncoder # Impor lagi biar jelas

# Buat data demo kecil
demo_df = pd.DataFrame({'Street': ['Pave', 'Grvl', 'Pave', 'Pave']})

# BEFORE
print("--- BEFORE ---")
print(demo_df)

# Gunakan transformer
ohe = OneHotEncoder(variables=['Street'], drop_last=False)
data_transformed = ohe.fit_transform(demo_df)

# AFTER
print("\n--- AFTER ---")
# Kolom 'Street' dipecah jadi 'Street_Pave' dan 'Street_Grvl' (isi 0 atau 1)
print(data_transformed)

--- BEFORE ---
  Street
0   Pave
1   Grvl
2   Pave
3   Pave

--- AFTER ---
   Street_Pave  Street_Grvl
0            1            0
1            0            1
2            1            0
3            1            0


In [ ]:
# BAGIAN B: PIPELINE PENUH (Untuk Produksi)

# Oke, sekarang kita gunakan semua fungsi tadi (dan beberapa lagi) untuk memproses **SEMUA DATA** secara otomatis pakai `Pipeline`.
# Ini adalah kode yang akan memproses `train.csv` dan `test.csv` lalu menyimpannya ke folder `output/`.

In [11]:
# 1. Impor semua library yang dibutuhkan
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

# Impor SEMUA transformer yg kita butuhkan
from feature_engine.imputation import (
    CategoricalImputer, 
    MeanMedianImputer,
    ArbitraryNumberImputer
)
from feature_engine.encoding import (
    RareLabelEncoder, 
    OneHotEncoder
)

print('Library untuk pipeline penuh siap.')

Library untuk pipeline penuh siap.


In [16]:
# 2. Load Data Mentah
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

# 3. Pisahkan Fitur (X) dan Target (y)
X_train = train_df.drop(['Id', 'SalePrice'], axis=1)
X_test = test_df.drop('Id', axis=1)

y_train_log = np.log1p(train_df['SalePrice'])

# --- TAMBAHKAN KODE INI ---
# PERBAIKAN: Ubah Tipe Data
# Kolom ini adalah angka, tapi sebenarnya kategori.
# Kita ubah jadi 'object' (string) SEBELUM masuk pipeline.
cols_to_cast = ['MSSubClass', 'OverallCond', 'YrSold', 'MoSold']
for col in cols_to_cast:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)
print("Tipe data berhasil diubah (cast).")
# -----------------------------

print(f"X_train asli: {X_train.shape}")
print(f"X_test asli: {X_test.shape}")

Tipe data berhasil diubah (cast).
X_train asli: (1460, 79)
X_test asli: (1459, 79)


In [17]:
# 4. Definisikan Grup Fitur (Ini bagian "ribet"-nya, tapi cuma 1x)

# Kolom numerik yang 'NA'-nya diisi median
FEATURES_NUM_MEDIAN = ['LotFrontage', 'GarageYrBlt']

# Kolom kategorial yang 'NA'-nya diisi string 'Missing'
FEATURES_CAT_MISSING = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]

# Kolom kategorial yang 'NA'-nya diisi modus (nilai terbanyak)
FEATURES_CAT_MODE = [
    'Electrical', 'KitchenQual', 'Functional', 'SaleType', 'Utilities', 'Exterior1st', 'Exterior2nd', 'MSZoning'
]

# Kumpulkan SEMUA kolom kategorial untuk di-encoding
# Kita juga masukkan kolom yg kelihatannya angka tapi sebenarnya kategori (MSSubClass)
FEATURES_CAT_ALL = FEATURES_CAT_MISSING + FEATURES_CAT_MODE + ['MSSubClass', 'OverallCond', 'YrSold', 'MoSold']

# Kolom numerik sisanya (kita akan biarkan, tapi 'NA' harus diisi)
# Kita bisa ambil semua kolom numerik yg BUKAN target
FEATURES_NUM_ALL = [col for col in X_train.select_dtypes(include='number') 
                    if col not in FEATURES_NUM_MEDIAN]

In [18]:
# 5. Buat Pipeline (Jauh lebih simpel dari 100 baris kode manual)
house_price_pipeline = Pipeline([

    # --- Tahap 1: Imputasi (Isi data kosong) ---
    ('num_imputer_median', MeanMedianImputer(imputation_method='median', variables=FEATURES_NUM_MEDIAN)),
    ('cat_imputer_missing', CategoricalImputer(imputation_method='missing', fill_value='Missing', variables=FEATURES_CAT_MISSING)),
    ('cat_imputer_mode', CategoricalImputer(imputation_method='frequent', variables=FEATURES_CAT_MODE)),
    
    # Isi sisa NaN numerik dengan 0 (contoh: 'GarageCars' yg NaN = 0 mobil)
    ('num_imputer_zero', ArbitraryNumberImputer(arbitrary_number=0, variables=FEATURES_NUM_ALL)),
    
    # --- Tahap 2: Encoding (Ubah jadi angka) ---
    ('rare_label_encoder', RareLabelEncoder(tol=0.01, n_categories=1, variables=FEATURES_CAT_ALL)),
    ('one_hot_encoder', OneHotEncoder(drop_last=True, variables=FEATURES_CAT_ALL)),
])

print("Pipeline preprocessing berhasil dibuat.")

Pipeline preprocessing berhasil dibuat.


In [19]:
# 6. Jalankan Pipeline
print("Fitting pipeline...")
# Pipeline BELAJAR dari data train
house_price_pipeline.fit(X_train)

print("Transforming train and test data...")
# Pipeline MENGUBAH data train dan test
X_train_processed = house_price_pipeline.transform(X_train)
X_test_processed = house_price_pipeline.transform(X_test)

print("Data berhasil diproses.")
print(f"Bentuk X_train baru: {X_train_processed.shape}")
print(f"Bentuk X_test baru: {X_test_processed.shape}")

Fitting pipeline...
Transforming train and test data...
Data berhasil diproses.
Bentuk X_train baru: (1460, 179)
Bentuk X_test baru: (1459, 179)


In [21]:
# 7. Simpan Hasil ke `output/`
# Gabungkan kembali X_train_processed dengan y_train_log
train_processed_final = X_train_processed.copy()
train_processed_final['SalePrice_Log'] = y_train_log

# Simpan ke CSV
train_processed_final.to_csv('output/train_processed.csv', index=False)
X_test_processed.to_csv('output/test_processed.csv', index=False)

print("File 'train_processed.csv' dan 'test_processed.csv' berhasil disimpan di folder 'output/'.")
print("\nSiap untuk 'train.ipynb'!")

File 'train_processed.csv' dan 'test_processed.csv' berhasil disimpan di folder 'output/'.

Siap untuk 'train.ipynb'!
